In [10]:
import sys, os
sys.path.append(os.path.abspath("../../"))
from src.dataset.data_loader import load_competition_data

train_dataset, test_dataset, sample_submission = load_competition_data()
print("Train shape:", train_dataset.shape)
print("Test shape :", test_dataset.shape)

Train shape: (2000, 7)
Test shape : (500, 7)


In [11]:
import torch, warnings, logging
from datasets import load_dataset

from transformers import pipeline as hf_pipeline
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM
)
from sentence_transformers import SentenceTransformer, util

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPTIONS = ['A', 'B', 'C', 'D', 'E']
print(f"Device : {DEVICE}")

Device : cpu


In [12]:
# HuggingFace Hub download bars + auth warning
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

# datasets library (.map(), load_dataset) progress bars
from datasets.utils.logging import disable_progress_bar as ds_disable_progress_bar
ds_disable_progress_bar()

# transformers logging (loading model/config messages)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# general warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# -------- Q1: Load train.csv with HF datasets (NOT pandas), build combined_text --------
hf_train = load_dataset(
    "csv",
    data_files="D:/DIPLOMA/PROJECTS/Genai_Project/genAI-project-24f1002692/data/train.csv"
)["train"]    # extract the train dataset using the train key from the dictionary returned from load_dataset

def add_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

hf_train = hf_train.map(add_combined_text)   # map calls the add_combined_text function for every row of hf_train
 
q1_answer = len(hf_train[51]["combined_text"])
print("=== Q1: len(combined_text) for row index 51 ===")
print(f"combined_text[51]: {hf_train[51]['combined_text']}")
print(f"Character length (Q1 answer): {q1_answer}")


# -------- Q2: bert-base-uncased tokenizer vocab size --------
bert_uncased_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
q2_answer = bert_uncased_tokenizer.vocab_size
print(f"\n=== Q2: bert-base-uncased vocab size ===")
print(f"Vocab size (Q2 answer): {q2_answer}")


print(' ')
# -------- Q3: [SEP] token ID --------
q3_answer = bert_uncased_tokenizer.sep_token_id
print("=== Q3: [SEP] token ID ===")
print(f"[SEP] token ID (Q3 answer): {q3_answer}")

# -------- Q4: Shape of input_ids after batch-tokenizing entire prompt column --------
all_prompts_q4 = [str(p) for p in hf_train["prompt"]]
encoded_q4 = bert_uncased_tokenizer(
    all_prompts_q4,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

q4_answer = tuple(encoded_q4["input_ids"].shape)
print(f"\n=== Q4: Shape of input_ids tensor ===")
print(f"input_ids shape (Q4 answer): {q4_answer}")
print(' ')

# ------ BERT / ROBERTA Architecture and Attention Mechanisms ----------
# -------- Q5: Size of each attention head (hidden_size / num_heads) --------
hidden_size_q5 = 768
num_heads_q5 = 12
q5_answer = hidden_size_q5 // num_heads_q5
print("=== Q5: Attention head dimensionality ===")
print(f"{hidden_size_q5} / {num_heads_q5} = {q5_answer} (Q5 answer)")

# -------- Q6: last_hidden_state shape for row ID 0 prompt (default tokenizer settings) --------
bert_uncased_model = AutoModel.from_pretrained("bert-base-uncased")
bert_uncased_model.eval()

row0_prompt = hf_train[0]["prompt"]
enc_row0 = bert_uncased_tokenizer(row0_prompt, return_tensors="pt")   # (1, 31) 'single prompt row 0'

with torch.no_grad():
    out_row0 = bert_uncased_model(**enc_row0)

q6_answer = tuple(out_row0.last_hidden_state.shape)
print(f"\n=== Q6: last_hidden_state shape (row ID 0 prompt) ===")
print(f"Token count : {enc_row0['input_ids'].shape}")
print(f"Shape (Q6 answer): {q6_answer}")
print(' ')

# -------- Q7: Sum of first 5 float values of the [CLS] embedding --------
cls_vec_row0 = out_row0.last_hidden_state[0, 0, :]   # [CLS] is token index 0
q7_answer = round(cls_vec_row0[:5].sum().item(), 4)
print("=== Q7: Sum of first 5 [CLS] vector values ===")
print(f"First 5 values : {cls_vec_row0[:5].tolist()}")
print(f"Sum (Q7 answer): {q7_answer}")

# -------- Q8: Attention weight from [CLS] to "fusion", last layer, head 0 --------
bert_attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_attn_model.eval()

sentence_q8 = "Light-ion fusion is a technique."
enc_q8 = bert_uncased_tokenizer(sentence_q8, return_tensors="pt")

with torch.no_grad():
    out_q8 = bert_attn_model(**enc_q8)

tokens_q8 = bert_uncased_tokenizer.convert_ids_to_tokens(enc_q8["input_ids"][0])
fusion_idx = tokens_q8.index("fusion")   # get the index of the fusion word

last_layer_attn = out_q8.attentions[-1]        # (batch, num_heads, seq, seq)
head0_attn = last_layer_attn[0, 0]             # (seq, seq) — first head

q8_answer = round(head0_attn[0, fusion_idx].item(), 4)
print(f"\n=== Q8: [CLS] -> 'fusion' attention weight (last layer, head 0) ===")
print(f"Tokens          : {tokens_q8}")
print(f"'fusion' index  : {fusion_idx}")
print(f"Attention weight (Q8 answer): {q8_answer}")

# -------- Q9: Cosine similarity (prompt vs Option B, row ID 0) using MiniLM --------
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

row0_q9 = hf_train[0]
prompt_emb_q9 = minilm_model.encode(row0_q9["prompt"], convert_to_tensor=True)
optionB_emb_q9 = minilm_model.encode(row0_q9["B"], convert_to_tensor=True)

q9_answer = round(util.cos_sim(prompt_emb_q9, optionB_emb_q9).item(), 4)
print("=== Q9: MiniLM cosine similarity — prompt vs Option B (row ID 0) ===")
print(f"Prompt   : {row0_q9['prompt']}")
print(f"Option B : {row0_q9['B']}")
print(f"Cosine similarity (Q9 answer): {q9_answer}")

=== Q1: len(combined_text) for row index 51 ===
combined_text[51]: Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character length (Q1 answer): 614

=== Q2: bert-base-uncased vocab size ===
Vocab size (Q2 answer): 30522
 
=== Q3: [SEP] token ID ===
[SEP] token ID (Q3 answer): 102

=== Q4: Shape of input_ids tensor ===
input_ids shape (Q4 answer): (2000, 128)
 
=== Q5: Attention head dimensionality ===
768 / 12 = 64 (Q5 answer)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2524.05it/s]



=== Q6: last_hidden_state shape (row ID 0 prompt) ===
Token count : torch.Size([1, 31])
Shape (Q6 answer): (1, 31, 768)
 
=== Q7: Sum of first 5 [CLS] vector values ===
First 5 values : [-0.4676641821861267, -0.07544481009244919, -0.20190081000328064, -0.007064245641231537, -0.4480220377445221]
Sum (Q7 answer): -1.2001


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3266.57it/s]



=== Q8: [CLS] -> 'fusion' attention weight (last layer, head 0) ===
Tokens          : ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
'fusion' index  : 4
Attention weight (Q8 answer): 0.1025


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2348.69it/s]


=== Q9: MiniLM cosine similarity — prompt vs Option B (row ID 0) ===
Prompt   : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Option B : Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Cosine similarity (Q9 answer): 0.7658


In [ ]:
OPTIONS = ['A', 'B', 'C', 'D', 'E']
train_q10 = train_dataset.copy()

# ---- Pipeline 1: TF-IDF cosine similarity (Milestone 1 approach) ----
all_texts_q10 = []
for col in ['prompt'] + OPTIONS:
    all_texts_q10.extend(train_q10[col].tolist())

tfidf_vec_q10 = TfidfVectorizer(stop_words='english')
tfidf_vec_q10.fit(all_texts_q10)

def tfidf_top3(row):
    texts = [row['prompt']] + [row[o] for o in OPTIONS]
    vecs = tfidf_vec_q10.transform(texts)    # fit on all the text and then transform row-wise
    prompt_vec = vecs[0]
    sims = {o: cosine_similarity(prompt_vec, vecs[i + 1])[0][0] for i, o in enumerate(OPTIONS)}
    return sorted(sims, key=sims.get, reverse=True)[:3]

train_q10['tfidf_top3_pred'] = train_q10.apply(tfidf_top3, axis=1)

# ---- Pipeline 2: all-MiniLM-L6-v2 embeddings for prompt + all 5 options ----
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

print("Encoding prompts with MiniLM...")
prompt_emb_all = minilm_model.encode(
    train_q10['prompt'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True
)

option_emb_all = {}
for opt in OPTIONS:
    print(f"Encoding option {opt}...")
    option_emb_all[opt] = minilm_model.encode(
        train_q10[opt].tolist(), batch_size=64, show_progress_bar=False, convert_to_numpy=True
    )

def minilm_top3(i):
    p_vec = prompt_emb_all[i]
    sims = {
        opt: cosine_similarity(p_vec.reshape(1, -1), option_emb_all[opt][i].reshape(1, -1))[0][0]
        for opt in OPTIONS
    }
    return sorted(sims, key=sims.get, reverse=True)[:3]

train_q10['minilm_top3_pred'] = [minilm_top3(i) for i in range(len(train_q10))]

def apk3(actual, predicted):
    if actual not in predicted:
        return 0.0
    return 1.0 / (predicted.index(actual) + 1)

# -------- Q10: MAP@3 of the MiniLM pipeline across the entire training set --------
q10_answer = train_q10.apply(lambda row: apk3(row['answer'], row['minilm_top3_pred']), axis=1).mean()
print(f"\n=== Q10: MiniLM pipeline MAP@3 (entire train set) ===")
print(f"MAP@3 (Q10 answer): {q10_answer:.4f}")

# -------- Q11: count where correct answer NOT in TF-IDF top3 BUT IS in MiniLM top3 --------
q11_answer = sum(
    (row['answer'] not in row['tfidf_top3_pred']) and (row['answer'] in row['minilm_top3_pred'])
    for _, row in train_q10.iterrows()
)
print(f"\n=== Q11: TF-IDF misses, MiniLM catches (count) ===")
print(f"Count (Q11 answer): {q11_answer}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5516.57it/s]


Encoding prompts with MiniLM...


Batches: 100%|██████████| 32/32 [00:18<00:00,  1.72it/s]


Encoding option A...
Encoding option B...
Encoding option C...
Encoding option D...
Encoding option E...


In [ ]:
zs_pipe_q12 = hf_pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

row1_q12 = hf_train[1]   # 2nd row = index 1
candidate_labels_q12 = [row1_q12['A'], row1_q12['B'], row1_q12['C']]

# -------- Q12: top-ranked option probability (default softmax) --------
result_q12 = zs_pipe_q12(row1_q12['prompt'], candidate_labels_q12)
q12_answer = round(result_q12['scores'][0], 4)   # candidate with max score is at 0th index
print("=== Q12: Zero-shot top-ranked probability (softmax) ===")
print(f"Prompt  : {row1_q12['prompt']}")
print(f"Labels  : {result_q12['labels']}")
print(f"Scores  : {[round(s, 4) for s in result_q12['scores']]}")
print(f"Top score (Q12 answer): {q12_answer}")

# -------- Q13: absolute difference in summed probabilities, softmax vs independent sigmoids --------
result_q13 = zs_pipe_q12(row1_q12['prompt'], candidate_labels_q12, multi_label=True)
sum_softmax = sum(result_q12['scores'])
sum_sigmoid = sum(result_q13['scores'])
q13_answer = round(abs(sum_softmax - sum_sigmoid), 4)
print(f"\n=== Q13: |sum(softmax) - sum(sigmoid)| ===")
print(f"Sum (softmax, multi_label=False) : {sum_softmax:.4f}")
print(f"Sum (sigmoid, multi_label=True)  : {sum_sigmoid:.4f}")
print(f"Absolute difference (Q13 answer) : {q13_answer}")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4555.15it/s]


=== Q12: Zero-shot top-ranked probability (softmax) ===
Prompt  : What is accelerator-based light-ion fusion?
Labels  : ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve partic

In [ ]:
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
flan_model.eval()
if torch.cuda.is_available():
    flan_model = flan_model.to(DEVICE)

row0_q14 = hf_train[0]
prompt_q14 = (
    f"Question: {row0_q14['prompt']}. "
    f"Is the correct answer A: {row0_q14['A']} or B: {row0_q14['B']}? "
    f"Answer with just the letter A or B."
)

inputs_q14 = flan_tokenizer(prompt_q14, return_tensors="pt")
if torch.cuda.is_available():
    inputs_q14 = {k: v.to(DEVICE) for k, v in inputs_q14.items()}

with torch.no_grad():
    output_ids_q14 = flan_model.generate(**inputs_q14, max_new_tokens=5)

q14_answer = flan_tokenizer.decode(output_ids_q14[0], skip_special_tokens=True)

print("=== Q14: flan-t5-small generated output ===")
print(f"Input prompt : {prompt_q14}")
print(f"Output (Q14 answer): \"{q14_answer}\"")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3911.43it/s]


=== Q14: flan-t5-small generated output ===
Input prompt : Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.
Output (Q14 answer): "B"


In [ ]:
print("MILESTONE 2 Q/A — ANSWERS SUMMARY")
print("=" * 55)
print(f"  Q1  — len(combined_text) row 51 : {q1_answer}")
print(f"  Q2  — bert-base-uncased vocab size : {q2_answer}")
print(f"  Q3  — [SEP] token ID : {q3_answer}")
print(f"  Q4  — input_ids shape (batch tokenize) : {q4_answer}")
print(f"  Q5  — attention head dimensionality : {q5_answer}")
print(f"  Q6  — last_hidden_state shape (row 0) : {q6_answer}")
print(f"  Q7  — sum first 5 [CLS] values : {q7_answer}")
print(f"  Q8  — attention weight [CLS] -> fusion : {q8_answer}")
print(f"  Q9  — MiniLM cosine sim (prompt vs B, row 0) : {q9_answer}")
print(f"  Q10 — MiniLM pipeline MAP@3 : {q10_answer:.4f}")
print(f"  Q11 — TF-IDF miss / MiniLM catch count : {q11_answer}")
print(f"  Q12 — Zero-shot top prob (softmax) : {q12_answer}")
print(f"  Q13 — |sum(softmax) - sum(sigmoid)| : {q13_answer}")
print(f"  Q14 — flan-t5-small generated output : \"{q14_answer}\"")

MILESTONE 2 Q/A — ANSWERS SUMMARY
  Q1  — len(combined_text) row 51 : 614
  Q2  — bert-base-uncased vocab size : 30522
  Q3  — [SEP] token ID : 102
  Q4  — input_ids shape (batch tokenize) : (2000, 128)
  Q5  — attention head dimensionality : 64
  Q6  — last_hidden_state shape (row 0) : (1, 31, 768)
  Q7  — sum first 5 [CLS] values : -1.2001
  Q8  — attention weight [CLS] -> fusion : 0.1025
  Q9  — MiniLM cosine sim (prompt vs B, row 0) : 0.7658
  Q10 — MiniLM pipeline MAP@3 : 0.4231
  Q11 — TF-IDF miss / MiniLM catch count : 462
  Q12 — Zero-shot top prob (softmax) : 0.4575
  Q13 — |sum(softmax) - sum(sigmoid)| : 0.9995
  Q14 — flan-t5-small generated output : "B"
